# Lesson 5 — Build a conditional VAE

A conditional variational autoencoder (CVAE) encodes a condition and target into a latent distribution. A decoder uses the condition and a latent sample to predict the target. For robot learning, the target is an action chunk. The condition can be robot state and camera features.

You will implement an encoder, the reparameterization operation, a conditional decoder, and the training and inference paths. Basic `nn.Linear` and activation layers are supplied by PyTorch. The exercises use small vectors first; the final experiment uses prepared ACT demonstration chunks. It does not download data.

The ELBO objective combines reconstruction error and KL divergence. Use the action mask from Lesson 4 for padded chunks. Reading: [Auto-Encoding Variational Bayes](https://arxiv.org/abs/1312.6114), plus the [ACT paper](https://arxiv.org/abs/2304.13705) for its later Transformer posterior.

**PyTorch note:** `torch.randn_like(mu)` makes a random tensor with the same shape and device. Multiplication and addition keep gradients, so the encoder learns through the sample.

The full answer notebook is [05-act-cvae-reference.ipynb](05-act-cvae-reference.ipynb).


# Before you code

This course assumes strong Python skills and basic machine-learning knowledge. We explain PyTorch when it first appears. Read each shape, unit, mask, and parameter definition before you run the check. The reference copy gives a complete solution; the learner copy leaves the implementation for you.

A conditional variational autoencoder (CVAE) represents uncertainty with a latent variable $z$. The encoder reads a condition $c$ and a training target $y$, and returns a mean and a log variance. Reparameterization turns them into a differentiable sample:

$$z=\mu(c,y)+e^{\ell(c,y)/2}\odot\varepsilon, \qquad \varepsilon\sim\mathcal{N}(0,I)$$

Where:
* $c$ - the condition, shape $(B,C)$; for ACT, observation features.
* $y$ - the training target, shape $(B,Y)$; for ACT, an action chunk.
* $\mu(c,y)$ - posterior mean, shape $(B,L)$.
* $\ell(c,y)$ - posterior log variance $\log\sigma^2$, shape $(B,L)$; $e^{\ell/2}=\sigma$ is the standard deviation.
* $\varepsilon$ - standard normal noise, shape $(B,L)$.
* $\odot$ - element-wise multiplication.
* $B$ - batch size; $C$, $Y$ - condition and target widths; $L$ - latent width.

During inference, the target is unavailable, so ACT sets $z$ to zero.

Worked scalar example: if $\mu=1$, $\ell=\log 4$, and $\varepsilon=0.5$, then $e^{\ell/2}=2$ and $z=1+2\cdot0.5=2$. The variance is 4, not the standard deviation; this is why the exponent uses half of log variance. The decoder combines c and z to predict y. ACT uses observations and demonstrated actions as its training condition and predicts an action chunk. Padding masks keep padded action tokens out of the posterior encoder. This lesson builds the CVAE idea with small PyTorch modules before the Transformer version.


In [ ]:
import os
import sys
from collections.abc import Iterable, Mapping
from functools import partial
from pathlib import Path


def find_workspace() -> Path:
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "manifests" / "act_references.yaml").is_file():
            return candidate
    raise FileNotFoundError("Start Jupyter inside this repository.")


WORKSPACE = find_workspace()
sys.path.insert(0, str(WORKSPACE / "src"))
os.environ.setdefault("HF_HUB_CACHE", str(WORKSPACE / ".local-data/hf/hub"))
os.environ.setdefault("HF_DATASETS_CACHE", str(WORKSPACE / ".local-data/hf/datasets"))
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("HF_DATASETS_OFFLINE", "1")
os.environ.setdefault("MPLCONFIGDIR", str(WORKSPACE / ".local-data/matplotlib"))
import matplotlib.pyplot as plt
import torch
from torch import nn

print("Workspace:", WORKSPACE)
print("PyTorch:", torch.__version__)

In [ ]:
torch.manual_seed(4)
CONDITION_DIM, TARGET_DIM, LATENT_DIM, HIDDEN = 3, 2, 2, 8
condition = torch.randn(5, CONDITION_DIM)
target = torch.randn(5, TARGET_DIM)

## Exercises

Implement one component at a time.

## Exercise 1 — Conditional encoder

Implement `ConditionalEncoder`. Join condition and target on the last axis. The network returns `mu` and `logvar`, each with shape `(..., latent_dim)`. The exercise fixture uses `(B,C)` and `(B,Y)` inputs.

**Method:** map the concatenated values through a small shared network, then use separate linear heads for mean and log variance. Clamp `logvar` to a stable range.

**Pointers:** `torch.cat(..., dim=-1)` joins features while preserving the batch axis.

<details><summary>More hints</summary>

Use two hidden linear layers with GELU. Set input width to `condition_dim + target_dim`; set both heads to `latent_dim`.

</details>

More behaviour checks for this exercise: `check_conditional_encoder` in [lesson05_checks.py](lesson05_checks.py).

In [ ]:
class ConditionalEncoder(nn.Module):
    def __init__(
        self,
        condition_dim: int,
        target_dim: int,
        latent_dim: int,
        hidden_dim: int = 128,
        mask_dim: int = 0,
    ) -> None:
        super().__init__()
        raise NotImplementedError("Exercise 1: build the conditional encoder")

    def forward(
        self, condition: torch.Tensor, target: torch.Tensor, target_mask: torch.Tensor | None = None
    ) -> tuple[torch.Tensor, torch.Tensor]:
        raise NotImplementedError("Exercise 1: return mu and logvar")

In [ ]:
encoder = ConditionalEncoder(CONDITION_DIM, TARGET_DIM, LATENT_DIM, HIDDEN)
mu, logvar = encoder(condition, target)
assert mu.shape == logvar.shape == (5, LATENT_DIM), (
    "Both Gaussian parameter outputs need shape (B, latent_dim)."
)
assert torch.isfinite(logvar).all(), "Log variance must be finite."
print("Encoder output shapes:", mu.shape, logvar.shape)

# More behaviour checks: lesson05_checks.py, next to this notebook.
from lesson05_checks import check_conditional_encoder

check_conditional_encoder(ConditionalEncoder)

In [ ]:
print("Use: the encoder output feeds the reparameterization function in Exercise 2.")

## Exercise 2 — Reparameterization

Implement `reparameterize(mu, logvar, noise=None)`. It returns a sample with shape `(B,L)`. When noise is supplied, use it instead of drawing random values. This makes the check reproducible.

$$z=\mu+e^{\frac{1}{2}\log\sigma^2}\odot\varepsilon$$

Where:
* $\mu$ - `mu`, shape $(B,L)$.
* $\log\sigma^2$ - `logvar`, shape $(B,L)$.
* $\varepsilon$ - `noise`, shape $(B,L)$; drawn from $\mathcal{N}(0,I)$ when not supplied.

**Pointers:** Keep `mu` and `logvar` in the calculation graph.

<details><summary>More hints</summary>

If `noise is None`, call `torch.randn_like(mu)`. Check the provided noise shape before using it.

</details>

More behaviour checks for this exercise: `check_reparameterize` in [lesson05_checks.py](lesson05_checks.py).

In [ ]:
def reparameterize(
    mu: torch.Tensor, logvar: torch.Tensor, *, noise: torch.Tensor | None = None
) -> torch.Tensor:
    raise NotImplementedError("Exercise 2: implement the differentiable sample")

In [ ]:
mu_check = torch.tensor([[1.0, -1.0]], requires_grad=True)
logvar_check = torch.zeros_like(mu_check, requires_grad=True)
z = reparameterize(mu_check, logvar_check, noise=torch.tensor([[2.0, 0.5]]))
torch.testing.assert_close(z, torch.tensor([[3.0, -0.5]]))
z.sum().backward()
assert torch.equal(mu_check.grad, torch.ones_like(mu_check)), (
    "The sample must keep its gradient to mu."
)
assert torch.equal(logvar_check.grad, torch.tensor([[1.0, 0.25]])), (
    "The sample must keep its gradient to logvar."
)
print("Sample:", z.detach())

# More behaviour checks: lesson05_checks.py, next to this notebook.
from lesson05_checks import check_reparameterize

check_reparameterize(reparameterize)

In [ ]:
print(
    "Use: the training path samples from the encoder; inference samples from a standard normal prior."
)

## Exercise 3 — Conditional decoder

Implement `ConditionalDecoder`. It takes condition `(..., C)` and latent `(..., L)` and returns a prediction `(..., Y)`. The output width is the target width.

**Pointers:** Join the condition and latent code along the final dimension. Keep all leading dimensions.

<details><summary>More hints</summary>

A two-layer MLP with GELU is enough for this exercise. The last layer width is `target_dim`.

</details>

More behaviour checks for this exercise: `check_conditional_decoder` in [lesson05_checks.py](lesson05_checks.py).

In [ ]:
class ConditionalDecoder(nn.Module):
    def __init__(
        self, condition_dim: int, latent_dim: int, target_dim: int, hidden_dim: int = 128
    ) -> None:
        super().__init__()
        raise NotImplementedError("Exercise 3: build the conditional decoder")

    def forward(self, condition: torch.Tensor, latent: torch.Tensor) -> torch.Tensor:
        raise NotImplementedError("Exercise 3: predict the target")

In [ ]:
decoder = ConditionalDecoder(CONDITION_DIM, LATENT_DIM, TARGET_DIM, HIDDEN)
prediction = decoder(condition, torch.zeros(5, LATENT_DIM))
assert prediction.shape == target.shape, "Decoder output must match the target shape."
assert torch.isfinite(prediction).all(), "Decoder output must be finite."
print("Decoder output:", prediction.shape)

# More behaviour checks: lesson05_checks.py, next to this notebook.
from lesson05_checks import check_conditional_decoder

check_conditional_decoder(ConditionalDecoder)

In [ ]:
print("Use: Exercise 4 combines encoder, sample, and decoder into a CVAE model.")

## Exercise 4 — Training and prior inference paths

Implement `ConditionalVAE`. `forward(condition, target, noise=...)` uses the target to form a posterior sample and returns `(prediction, mu, logvar)`. `infer(condition, noise=...)` samples from the standard normal prior and decodes without a target. For action chunks, set `target_shape=(T,D)` and pass `is_pad` to `forward`; zero padded target values before encoding and provide the time mask to the encoder. The caller can supply noise for a repeatable check.

**Pointers:** Keep posterior training and prior inference as separate methods. Store `latent_dim`.

<details><summary>More hints</summary>

In `forward`, call your encoder, reparameterize, and decode. In `infer`, create or accept a latent sample of shape `(*condition.shape[:-1], latent_dim)`.

</details>

More behaviour checks for this exercise: `check_conditional_vae` in [lesson05_checks.py](lesson05_checks.py).

In [ ]:
class ConditionalVAE(nn.Module):
    def __init__(
        self,
        condition_dim: int,
        target_dim: int,
        latent_dim: int,
        hidden_dim: int = 128,
        *,
        target_shape: tuple[int, ...] | None = None,
    ) -> None:
        super().__init__()
        raise NotImplementedError(
            "Exercise 4: assemble encoder and decoder; configure target_shape for chunks"
        )

    def forward(
        self,
        condition: torch.Tensor,
        target: torch.Tensor,
        *,
        noise: torch.Tensor | None = None,
        is_pad: torch.Tensor | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        raise NotImplementedError("Exercise 4: implement posterior path and mask padded values")

    def infer(self, condition: torch.Tensor, *, noise: torch.Tensor | None = None) -> torch.Tensor:
        raise NotImplementedError("Exercise 4: implement prior inference path")

In [ ]:
cvae = ConditionalVAE(CONDITION_DIM, TARGET_DIM, LATENT_DIM, HIDDEN)
train_prediction, mu, logvar = cvae(condition, target, noise=torch.zeros(5, LATENT_DIM))
prior_prediction = cvae.infer(condition, noise=torch.ones(5, LATENT_DIM))
assert train_prediction.shape == prior_prediction.shape == target.shape
assert mu.shape == logvar.shape == (5, LATENT_DIM)
assert not torch.equal(train_prediction, prior_prediction), (
    "Posterior training and prior inference should use their supplied latent values."
)
print("Training and inference paths have the expected shapes.")

# More behaviour checks: lesson05_checks.py, next to this notebook.
from lesson05_checks import check_conditional_vae

check_conditional_vae(ConditionalVAE)

In [ ]:
print("Use: the training path now has the same posterior/prior distinction used by a CVAE policy.")

## Connect the model to a training callback

Implement a callback with the Lesson 4 contract: `loss_fn(model, batch, *, noise=None, time=None) -> (scalar, metrics)`. Use masked action L1 for padded action chunks and Gaussian KL from Lesson 4. `batch` is a `dict[str, Tensor]` with these keys:
* `condition` - shape $(B,C)$.
* `target` - shape $(B,Y)$ for vectors, or $(B,T,D)$ for action chunks.
* `is_pad` - only for chunks: shape $(B,T)$, `True` for padded steps.

Return the scalar loss tensor and a metrics `dict` with the two parts as scalar tensors: `reconstruction` (masked L1) and `kl`.

**Pointers:** Keep `time` for callback compatibility. Pass `noise` to `model` so a check can be repeatable. The returned loss must keep gradients.

More behaviour checks for this exercise: `check_cvae_loss_fn` in [lesson05_checks.py](lesson05_checks.py).

In [ ]:
def cvae_loss_fn(
    model: ConditionalVAE,
    batch: Mapping[str, torch.Tensor],
    *,
    noise: torch.Tensor | None = None,
    time: torch.Tensor | None = None,
    beta: float = 1.0,
) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
    """Return reconstruction + beta*KL and scalar metrics."""
    raise NotImplementedError("Implement callback; pass batch is_pad into model")

In [ ]:
mini_model = ConditionalVAE(CONDITION_DIM, TARGET_DIM, LATENT_DIM, HIDDEN)
mini_batch = {"condition": condition, "target": target}
noise = torch.zeros(5, LATENT_DIM)
loss, metrics = cvae_loss_fn(mini_model, mini_batch, noise=noise, beta=0.1)
assert loss.ndim == 0 and torch.isfinite(loss), "Callback must return one finite scalar tensor."
loss.backward()
assert any(parameter.grad is not None for parameter in mini_model.parameters()), (
    "Callback loss must preserve the gradient path."
)
print("Callback metrics:", {name: float(value.detach()) for name, value in metrics.items()})
# Padded target values cannot change the posterior or objective when random noise is fixed.
chunk_model = ConditionalVAE(CONDITION_DIM, 10, LATENT_DIM, HIDDEN, target_shape=(5, 2))
chunk_mask = torch.tensor(
    [[0, 0, 0, 0, 0], [0, 0, 1, 1, 1], [0, 0, 0, 1, 1], [0, 0, 0, 0, 1], [0, 1, 1, 1, 1]],
    dtype=torch.bool,
)
chunk_data = {"condition": condition, "target": torch.randn(5, 5, 2), "is_pad": chunk_mask}
chunk_loss, _ = cvae_loss_fn(chunk_model, chunk_data, noise=noise, beta=0.1)
changed = dict(chunk_data, target=chunk_data["target"].clone())
changed["target"][chunk_mask] = 500
changed_loss, _ = cvae_loss_fn(chunk_model, changed, noise=noise, beta=0.1)
torch.testing.assert_close(
    chunk_loss, changed_loss, msg="Padded action values must not affect CVAE posterior or loss"
)
print("Chunk mask check passed.")

# More behaviour checks: lesson05_checks.py, next to this notebook.
from lesson05_checks import check_cvae_loss_fn

check_cvae_loss_fn(cvae_loss_fn)

Use: later units reuse this callback shape for ACT, diffusion, and flow objectives. A caller can pass explicit random values through `noise` and `time` for repeatable checks.

Run this cell to fit the small conditional VAE on local ACT demonstration chunks. This checks the latent-variable training path; it does not train the complete ACT policy or establish simulator success. Automated quick checks skip this experiment and report that skip.

In [ ]:
# Fit a small one-dimensional conditional example and inspect several prior samples.
torch.manual_seed(23)
toy_condition = torch.linspace(-2, 2, 96).unsqueeze(-1)
toy_target = torch.sin(1.5 * toy_condition) + 0.05 * torch.cos(4 * toy_condition)
toy_data = [
    {"condition": toy_condition[start : start + 16], "target": toy_target[start : start + 16]}
    for start in range(0, 96, 16)
]
toy_model = ConditionalVAE(1, 1, 2, hidden_dim=32)
toy_optimizer = torch.optim.Adam(toy_model.parameters(), lr=0.01)
from vla_lab.teaching.training import train_loop as reference_train_loop

toy_callback = partial(cvae_loss_fn, beta=0.05)
toy_history = reference_train_loop(toy_model, toy_data, toy_optimizer, toy_callback, steps=120)
toy_model.eval()
with torch.no_grad():
    prior_zero = toy_model.infer(toy_condition, noise=torch.zeros(96, 2))
    prior_one = toy_model.infer(toy_condition, noise=torch.ones(96, 2))
plt.scatter(toy_condition[:, 0], toy_target[:, 0], s=12, label="target")
plt.plot(toy_condition[:, 0], prior_zero[:, 0], label="prior noise = 0")
plt.plot(toy_condition[:, 0], prior_one[:, 0], label="prior noise = 1")
plt.xlabel("condition")
plt.ylabel("target")
plt.title("Small CVAE example: trained predictions")
plt.legend()
plt.show()
plt.plot([item["loss"] for item in toy_history])
plt.xlabel("Update")
plt.ylabel("Toy CVAE loss")
plt.show()
from vla_lab.teaching.live import load_act_batches

prepared = load_act_batches(
    "bowl_plate",
    WORKSPACE,
    batch_size=2,
    chunk_size=20,
    train_batches=2,
    validation_batches=1,
    seed=31,
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def adapt(rows: Iterable[Mapping[str, torch.Tensor]]) -> list[dict[str, torch.Tensor]]:
    return [
        {
            "condition": row["observation.state"].to(device),
            "target": row["action"].to(device),
            "is_pad": row["action_is_pad"].to(device),
        }
        for row in rows
    ]


train_rows, validation_rows = adapt(prepared.train), adapt(prepared.validation)
results = []
settings = ((4, 0.1), (8, 0.1), (4, 1.0))
for latent_width, beta in settings:
    row = {"latent_dim": latent_width, "beta": beta, "status": "failed"}
    optimizer = model = None
    try:
        torch.manual_seed(41)
        model = ConditionalVAE(8, 20 * 7, latent_width, hidden_dim=64, target_shape=(20, 7)).to(
            device
        )
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
        torch.manual_seed(1041)
        if device.type == "cuda":
            torch.cuda.manual_seed_all(1041)
        callback = partial(cvae_loss_fn, beta=beta)
        reference_train_loop(model, train_rows, optimizer, callback, steps=20)
        model.eval()
        with torch.no_grad():
            scores = [
                callback(
                    model,
                    row,
                    noise=torch.zeros(row["condition"].shape[0], latent_width, device=device),
                )[1]
                for row in validation_rows
            ]
        row["validation_reconstruction"] = sum(
            item["reconstruction"].item() for item in scores
        ) / len(scores)
        row["validation_kl"] = sum(item["kl"].item() for item in scores) / len(scores)
        row["status"] = "complete"
    except Exception as error:
        row["error"] = f"{type(error).__name__}: {error}"
    results.append(row)
    del optimizer, model
    if device.type == "cuda":
        torch.cuda.empty_cache()
print("Prepared split:", prepared.metadata)
for item in results:
    print(item)
print(
    "Compare latent width within beta 0.1 and compare beta within latent width 4. Keep failed rows."
)
print("The evaluation uses zero posterior noise. This short run does not show policy success.")

## What this model does not include

This small CVAE demonstrates posterior training and prior inference with a conditional MLP. Lesson 8 assembles ACT with a Transformer posterior over state and action tokens, camera position information, and action queries. It is a different architecture. Do not treat this MLP as an ACT implementation.